# Sesión 2.1 — Inferencia estadística

Estimación e inferencia sobre medias, varianzas y proporciones. Ejecuta las celdas en orden.

## Población, muestra y estimadores

La **población** es el conjunto sobre el que queremos concluir; la **muestra** contiene las observaciones disponibles. Un **parámetro** (por ejemplo, la media poblacional $\mu$ o la proporción $\pi$) describe la población, mientras que un **estadístico** calculado a partir de la muestra sirve para estimarlo. Como otra muestra produciría otros estadísticos, los intervalos de confianza y los contrastes cuantifican esa incertidumbre bajo sus supuestos.


## Variabilidad muestral y calidad de la muestra

Si repitiésemos el muestreo, la media y la proporción muestrales variarían de una muestra a otra. El **error estándar** estima esa variación: para la media, es aproximadamente $s/\sqrt{n}$. A igualdad de lo demás, una muestra mayor reduce el error estándar y estrecha el intervalo; pedir más confianza lo ensancha. Una mayor dispersión de los datos suele aumentar la incertidumbre.

Esto describe la **incertidumbre aleatoria**, pero no corrige una muestra sesgada. Para generalizar a una población, la muestra debe representarla razonablemente: por ejemplo, encuestar solo a clientes de una tienda o de un único plan podría dejar fuera sistemáticamente a otros perfiles. Aumentar el tamaño no elimina por sí solo ese sesgo.

Para esta práctica, asumimos que las muestras que nos han facilitado representan razonablemente la población a estudiar.

## 2.0. Lectura de datos

Abrimos la tabla de teléfonos móviles y la mostramos para poder inspeccionarla. Los campos están separados por `;`; `NA` y las celdas vacías representan valores ausentes.

In [ ]:
from pathlib import Path  # Permite construir rutas a los ficheros de forma portable.
import numpy as np  # Proporciona operaciones numéricas y generación de remuestras.
import pandas as pd  # Permite leer y manipular tablas de datos.
import matplotlib.pyplot as plt  # Sirve para crear gráficos.
from scipy import stats  # Aporta distribuciones y pruebas estadísticas.
from IPython.display import display  # Muestra tablas y resultados con formato en el notebook.

DATA_PATH = Path.cwd().parent.parent / "datos"  # Localiza la carpeta datos desde el directorio actual.

datos = pd.read_csv(DATA_PATH / 'Mobiles.txt', sep=';', na_values=['NA', ''])  # Lee el fichero separado por punto y coma y marca NA y celdas vacías como ausentes.
display(datos)  # Muestra la tabla completa para inspeccionar sus filas y columnas.

## 2.1. Estimación de la media poblacional

Primero examinamos la distribución numérica y gráficamente. La prueba de una muestra estima la media con su intervalo de confianza (IC) y contrasta la hipótesis de que la media poblacional sea 30 (referencia del PDF) o 50.

In [ ]:
# Resumen numérico de las facturas: cuenta, media, desviación, cuartiles, mínimo y máximo.
datos["Bill"].describe()  # Calcula las estadísticas descriptivas de la columna Bill.

### Qué nos dice el resumen descriptivo

- **Tamaño válido:** 174 facturas; el recuento excluye los valores ausentes.
- **Centro:** la media es 46,85 y la mediana, 35. La media queda por encima de la mediana porque los importes altos tiran de ella hacia la derecha.
- **Dispersión y extremos:** la desviación típica es 41,58, el mínimo es 0 y el máximo 300. La amplitud entre valores y la dispersión sugieren que los importes no se agrupan estrechamente alrededor de la media.

**Implicación para el análisis:** la media es una medida útil, pero no representa por sí sola el patrón típico de factura. Conviene acompañarla de la mediana y de gráficos; los valores extremos también pueden influir en las pruebas e intervalos basados en la media.

In [ ]:
# Histograma de las facturas: azul claro de la diapositiva y bordes gris oscuro.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura ancha y el eje donde dibujar.
ax.hist(datos['Bill'], bins='auto', color='#9fc3e4', edgecolor='#666666', linewidth=1.2)  # Agrupa las facturas en intervalos y muestra sus frecuencias.
ax.set_title('Distribución del importe de la factura', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Importe de la factura', fontsize=16)  # Etiqueta el eje horizontal con la variable medida.
ax.set_ylabel('Número de estudiantes', fontsize=16)  # Etiqueta el eje vertical con el recuento.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el histograma.

### Lectura del histograma

La mayor parte de las facturas se concentra aproximadamente entre 20 y 60. A partir de ahí aparece una **cola larga hacia la derecha**, con pocos importes muy altos, algunos próximos a 250–300. Por tanto, la distribución no es simétrica ni tiene la forma de campana que esperaríamos en una normal.

**Implicación:** esta forma explica por qué la media (46,85) supera a la mediana (35). Los valores altos no deben eliminarse automáticamente, podrían ser observaciones reales, pero sí deben tenerse en cuenta: elevan la media y pueden afectar a la inferencia. El histograma sugiere revisar la normalidad con el gráfico Q–Q antes de usar métodos que la requieren.

### 2.1.1 Intervalo de confianza para la media

Queremos estimar el importe medio de la factura en la población. Un IC bilateral con confianza $1-\alpha$ se calcula con la distribución $t$ de Student, porque la desviación típica poblacional es desconocida:

$$
IC_{1-\alpha}(\mu)=\bar{x}\pm t_{1-\alpha/2,\,n-1}\,\frac{s}{\sqrt{n}}
=\left[\bar{x}-t_{1-\alpha/2,\,n-1}\frac{s}{\sqrt{n}},\;\bar{x}+t_{1-\alpha/2,\,n-1}\frac{s}{\sqrt{n}}\right].
$$

Aquí $\bar{x}$ es la media muestral, $s$ la desviación típica muestral, $n$ el número de observaciones válidas y $s/\sqrt{n}$ el error estándar. Para un IC del 95 %, $\alpha=0{,}05$; para uno del 99 %, $\alpha=0{,}01$. Al aumentar la confianza, aumenta el cuantil crítico y el intervalo se ensancha.

La función descarta los valores ausentes y aplica esta fórmula. La cobertura del 95 % describe el comportamiento del procedimiento en muestreos repetidos: no significa que la media fija tenga un 95 % de probabilidad de estar en este intervalo concreto.

In [ ]:
def intervalo_media(serie, confianza=0.95):  # Define una función para estimar la media y su intervalo bilateral.
    """Intervalo de confianza bilateral para la media de una muestra."""  # Explica el propósito de la función.
    muestra = pd.Series(serie).dropna()  # Convierte la entrada en serie y elimina valores ausentes.
    ic = stats.t.interval(confianza, df=len(muestra)-1,  # Calcula los límites con t de Student y n−1 grados de libertad.
                          loc=muestra.mean(), scale=stats.sem(muestra))  # Usa la media muestral y su error estándar.
    return pd.Series({'n': len(muestra), 'media': muestra.mean(),  # Devuelve el tamaño y la media calculados con los datos válidos.
                      'IC inferior': ic[0], 'IC superior': ic[1]})  # Añade los límites inferior y superior del intervalo.

In [ ]:
# IC del 95 % para el importe medio de la factura.
display(intervalo_media(datos['Bill']))  # Aplica la función a Bill y presenta el resultado como tabla.

### Interpretación del IC de la media

La media observada es **46,85** y el intervalo de confianza del 95 % va de **40,63 a 53,07**. Bajo las condiciones del método, si repitiésemos el muestreo y construyésemos el intervalo del mismo modo, alrededor del 95 % de esos intervalos cubrirían la media poblacional.

**Qué permite concluir:** el intervalo resume incertidumbre acerca de la **media poblacional**; no afirma que el 95 % de las facturas individuales esté entre esos límites. El rango es compatible con valores alrededor de 50, pero no con 30, cuestión que se comprueba formalmente con los contrastes siguientes.

**Cautela:** la interpretación depende de un muestreo razonablemente representativo e independiente. Con $n=174$, la aproximación para la media puede ser útil, pero la fuerte asimetría y los valores extremos aconsejan leer el resultado junto con los gráficos.

#### Micropráctica: aumentar la confianza

Mira la función y el ejemplo del IC del 95 % justo arriba. Completa el hueco del código para calcular el IC del 99 %, ejecútalo y compara ambos intervalos. Antes de ejecutarlo, predice cuál será más ancho.

In [ ]:
try:
    # Calcula un IC del 99 % y compara su amplitud con la del 95 %.
    ic_95 = intervalo_media(datos['Bill'], confianza=0.95)  # Intervalo de referencia.
    confianza = ...  # Define el nuevo nivel de confianza.
    ic_nuevo = intervalo_media(datos['Bill'], confianza=...)  # Completa el argumento que falta.
    amplitud_95 = ic_95['IC superior'] - ic_95['IC inferior']  # Calcula la anchura del primer intervalo.
    amplitud_nueva = ...  # Calcula la anchura del nuevo intervalo usando sus límites.
    print(f'IC nuevo:\n {ic_nuevo} \n')  # Muestra los límites obtenidos.
    print(f'Amplitud 95 %: {amplitud_95:.2f}; amplitud nueva: {amplitud_nueva:.2f}')  # Compara ambas anchuras.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

### 2.1.2 Contraste de hipótesis para una media: prueba t de una muestra

El intervalo **estima** la media; un contraste pregunta si los datos son compatibles con un valor propuesto $\mu_0$. Planteamos una **hipótesis nula** frente a una alternativa bilateral:

$$
\left\{
\begin{aligned}
H_0 &: \mu = \mu_0 \\
H_1 &: \mu \ne \mu_0
\end{aligned}
\right.
$$

La prueba t compara la diferencia entre la media observada y $\mu_0$ con el error estándar de la media.

El **estadístico t** mide esa diferencia en unidades de error estándar. El **valor p** indica cuán extremos serían los datos observados (o más) si $H_0$ fuera cierta. Por ejemplo, con un nivel de significación de 0,05, rechazamos $H_0$ si $p<0{,}05$; si no, **no la rechazamos**, lo que no demuestra que sea verdadera. La prueba presupone observaciones independientes y, para muestras pequeñas, una distribución aproximadamente normal.

In [ ]:
def contraste_media(serie, mu=0):  # Define un contraste t bilateral frente al valor hipotético mu.
    """Contraste bilateral de la hipótesis de que la media poblacional es mu."""  # Resume la hipótesis que evalúa la función.
    muestra = pd.Series(serie).dropna()  # Conserva únicamente las observaciones no ausentes.
    resultado = stats.ttest_1samp(muestra, popmean=mu)  # Calcula el estadístico t y el valor p frente a mu.
    print(f"IC95: ({resultado.confidence_interval(0.99)[0]}, {resultado.confidence_interval(0.95)[1]})\n")
    return pd.Series({'n': len(muestra), 'media': muestra.mean(),  # Devuelve el tamaño y la media observada.
                      'media hipotética': mu, 't': resultado.statistic,  # Incluye el valor nulo y el estadístico de prueba.
                      'p-valor': resultado.pvalue})  # Incluye el valor p bilateral para interpretar el contraste.

#### Ejemplo 1: ¿podría ser 30 la media poblacional?

Contrastamos $H_0: \mu=30$ frente a $H_1: \mu\ne30$. El estadístico de la prueba t de una muestra es

$$
t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}},\qquad gl=n-1.
$$

Con $\alpha=0{,}05$ y 173 grados de libertad, las regiones de rechazo (los dos **bigotes**) son $t<-1{,}974$ y $t>1{,}974$. Compara la salida con esos límites y con $\alpha$. El valor 30 se toma como referencia didáctica del PDF; los datos y las unidades de este ejemplo son distintos.

In [ ]:
display(contraste_media(datos['Bill'], mu=30))  # Ejecuta el contraste H₀: media poblacional = 30 y muestra sus resultados.

### Interpretación formal del contraste

La salida da $t_{obs}=5{,}346$ y $p\approx2{,}8\times10^{-7}$. El estadístico supera el límite crítico superior ($1{,}974$) y el valor p es menor que $0{,}05$; cae en un **bigote o región de rechazo**. Por ambos criterios, **rechazamos $H_0: \mu=30$**.

La muestra aporta evidencia de que la media difiere de 30. Esto no determina por sí solo la importancia práctica de la diferencia ni corrige posibles problemas de representatividad; la conclusión también requiere observaciones independientes y que el método sea adecuado para los datos.

#### Ejemplo 2: ¿podría ser 50 la media poblacional?

Contrastamos $H_0: \mu=50$ frente a $H_1: \mu\ne50$. El estadístico de la prueba t de una muestra es

$$
t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}},\qquad gl=n-1.
$$

Con $\alpha=0{,}05$ y 173 grados de libertad, los **bigotes o regiones de rechazo** son $t<-1{,}974$ y $t>1{,}974$ (0,025 en cada cola). Compara el estadístico y el valor p de la salida con estos límites y con $\alpha$.

In [ ]:
display(contraste_media(datos['Bill'], mu=50))  # Ejecuta el contraste H₀: media poblacional = 50 y muestra sus resultados.

### Interpretación formal del contraste

La salida da $t_{obs}=-0{,}999$ y $p=0{,}3191$. El estadístico queda entre los valores críticos $-1{,}974$ y $1{,}974$; no cae en ninguno de los dos **bigotes o regiones de rechazo**. De forma equivalente, $p=0{,}3191>\alpha=0{,}05$. Por ambos criterios, **no rechazamos $H_0: \mu=50$**.

El signo negativo indica que la media muestral (46,85) está unos 3,15 euros por debajo del valor hipotético. Sin embargo, esa diferencia es de apenas un error estándar (aproximadamente $t=-1$), por lo que no es suficientemente extrema para el contraste bilateral al 5 %.

**Conclusión:** la muestra no aporta evidencia suficiente para afirmar que la media poblacional difiera de 50. Esto no demuestra que la media sea exactamente 50. La decisión concuerda con el intervalo de confianza del 95 % mostrado, que contiene 50. Como siempre, la conclusión supone observaciones independientes y un muestreo adecuado; la asimetría observada merece mencionarse al valorar la inferencia.

#### Micropráctica: elegir el nivel de significación

Contrasta $H_0: \mu=40$ frente a $H_1: \mu\ne40$ con la función anterior. Antes de ejecutar, elige y justifica un nivel de significación $\alpha$ (por ejemplo, 0,10, 0,05 o 0,01). Completa el código para que la decisión se base en tu elección y compara el estadístico observado con los valores críticos de ambas colas. ¿Cambiaría la decisión si eligieras otro $\alpha$?

In [ ]:
try:
    # Contrasta H₀: μ = 40 frente a una alternativa bilateral.
    mu_ref = 40  # Valor de referencia fijado para el ejercicio.
    resultado = contraste_media(datos['Bill'], mu=mu_ref)  # Calcula t y el valor p.
    display(resultado)  # Revisa el estadístico y el valor p.
    alpha = ...  # Elige y justifica tu nivel de significación (por ejemplo, 0.10, 0.05 o 0.01).
    if resultado['p-valor'] < alpha:  # Compara el valor p con el alfa elegido.
        print('Rechazamos H₀ al nivel de significación elegido.')
    else:
        print('No rechazamos H₀ al nivel de significación elegido.')
    # Opcional: compara |t| con stats.t.ppf(1 - alpha/2, df=resultado['n'] - 1).
    # Redacta la conclusión en contexto; no rechazar H₀ no demuestra que sea cierta.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

## 2.2 Estimación de la desviación poblacional

### 2.2.1a Intervalo basado en la distribución $\chi^2$

Si la población es normal, el intervalo bilateral de confianza $1-\alpha$ para la varianza poblacional $\sigma^2$ es

$$
IC_{1-\alpha}(\sigma^2)=\left[\frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}},\;\frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}}\right].
$$

Al tomar la raíz cuadrada de ambos límites se obtiene el intervalo para la desviación típica poblacional:

$$
IC_{1-\alpha}(\sigma)=\left[\sqrt{\frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}}},\;\sqrt{\frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}}}\right].
$$

Los cuantiles están en orden inverso en los denominadores, por eso los límites siguen siendo crecientes. **Este método requiere normalidad e independencia**; más adelante verificamos que la normalidad no parece adecuada para estas facturas.

### 2.2.1b Alternativa: bootstrap por percentiles

Se generan $B$ remuestras del mismo tamaño, con reemplazo, y se calcula una desviación típica $s_b^*$ en cada una. El IC percentil se forma con los cuantiles de esas estimaciones:

$$
IC_{1-\alpha}^{\mathrm{boot}}(\sigma)=\left[Q_{\alpha/2}(s_1^*,\ldots,s_B^*),\;Q_{1-\alpha/2}(s_1^*,\ldots,s_B^*)\right].
$$

Por ejemplo, para el 95 % se usan los cuantiles 0,025 y 0,975; para el 90 %, 0,05 y 0,95. Bootstrap no exige normalidad poblacional, pero sí que la muestra sea razonablemente representativa y las observaciones independientes. Fijamos una semilla para hacer reproducible la simulación.

In [ ]:
s2 = datos["Bill"].var(ddof=1)  # Calcula la varianza muestral de Bill usando n−1 en el denominador.
n = datos["Bill"].notna().sum()  # Cuenta cuántas facturas tienen un valor conocido.

print(f"Varianza muestral: {s2} \nTamaño muestral: {n}")  # Muestra la varianza y el número de observaciones válidas.

In [ ]:
chi2 = stats.chi2.ppf(0.025, n-1)  # Obtiene el cuantil 2,5 % de chi-cuadrado para n−1 grados de libertad.
chi1 = stats.chi2.ppf(0.975, n-1)  # Obtiene el cuantil 97,5 % para el límite opuesto del intervalo.
LI = (n-1)*s2/chi1  # Calcula el límite inferior del intervalo para la varianza.
LS = (n-1)*s2/chi2  # Calcula el límite superior del intervalo para la varianza.
display(pd.Series({'Varianza inferior': LI, 'Varianza superior': LS,  # Prepara una tabla con los dos límites de la varianza.
                   'Desviación inferior': np.sqrt(LI), 'Desviación superior': np.sqrt(LS)}))  # Añade las raíces, que son los límites para la desviación típica.

### Interpretación del intervalo basado en $\chi^2$

Con $n=174$, la varianza muestral es **1728,69**. El intervalo del 95 % resultante para la desviación típica va aproximadamente de **37,62 a 46,47**; para la varianza, de **1415,26 a 2159,74**.

**Supuesto clave:** este procedimiento exacto para la varianza presupone una población normal. Los gráficos y la prueba que siguen muestran que las facturas son muy asimétricas y no normales. Por eso, estos límites ilustran cómo se aplica la fórmula, pero no conviene presentarlos como una estimación fiable sin advertir que falla un supuesto central. Más adelante se usa bootstrap como alternativa que no exige normalidad poblacional.

#### Micropráctica: intervalo del 90 % para la varianza

Usa como modelo el cálculo del IC del 95 % de arriba. Completa el valor de $\alpha$ para una confianza del 90 %, ejecuta la celda y comprueba si el intervalo queda más estrecho.

In [ ]:
try:
    # Calcula el IC bilateral del 90 % para la varianza usando s2 y n de arriba.
    alpha_90 = ...  # Relaciona alfa con el nivel de confianza pedido.
    q_sup = stats.chi2.ppf(..., n-1)  # Cuantil alto de chi-cuadrado, con n−1 grados de libertad.
    q_inf = stats.chi2.ppf(..., n-1)  # Cuantil bajo de chi-cuadrado, con n−1 grados de libertad.
    li_var_90 = (n-1) * s2 / ...  # Calcula el límite inferior con el cuantil adecuado.
    ls_var_90 = (n-1) * s2 / ...  # Calcula el límite superior con el otro cuantil.
    print(f'IC del 90 % para la varianza: ({li_var_90:.2f}, {ls_var_90:.2f})')  # Presenta el intervalo para compararlo con el 95 %.
    # Comprueba que el intervalo del 90 % sea más estrecho que el del 95 %.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### 2.2.2 Contraste de una varianza: referencia $\sigma=8$ (Opcional)

La pregunta se formula sobre la **desviación típica poblacional** $\sigma$:

$$
\begin{aligned}
H_0 &: \sigma = 8 \\
H_1 &: \sigma \ne 8
\end{aligned}
$$

Como la prueba $\chi^2$ se expresa en términos de la varianza, la misma hipótesis se escribe de forma equivalente como $H_0: \sigma^2=64$ frente a $H_1: \sigma^2\ne64$. El estadístico es

$$
\chi^2_{obs}=\frac{(n-1)s^2}{\sigma_0^2},\qquad gl=n-1,
$$

con $\sigma_0=8$. Bajo normalidad e independencia, sigue una distribución $\chi^2$ bajo $H_0$. **La cifra 8 es la referencia didáctica del PDF**; este notebook usa otro conjunto de datos, así que el resultado numérico no es directamente comparable.

In [ ]:
# Contraste bilateral H₀: sigma² = 8²; referencia ilustrativa del PDF.
sigma0 = 8  # Define la desviación típica de referencia bajo H₀.
bill_validos = datos['Bill'].dropna()  # Selecciona las facturas observadas y elimina las ausentes.
n_bill = len(bill_validos)  # Cuenta las observaciones válidas.
chi2_observado = (n_bill - 1) * bill_validos.var(ddof=1) / sigma0**2  # Calcula el estadístico chi-cuadrado bajo H₀.
gl = n_bill - 1  # Fija los grados de libertad del contraste.
p_chi2 = 2 * min(stats.chi2.cdf(chi2_observado, gl), stats.chi2.sf(chi2_observado, gl))  # Calcula el valor p bilateral sumando la cola más extrema por dos.
print(f'χ² = {chi2_observado:.4f}; gl = {gl}; p-valor bilateral = {p_chi2:.7g}')  # Muestra el estadístico, grados de libertad y valor p.
print('Se rechaza H₀ al 5 %.' if p_chi2 < 0.05 else 'No se rechaza H₀ al 5 %.')  # Decide comparando el valor p con alfa = 0,05.

### Lectura del contraste de varianza

El estadístico observado es $\chi^2=4672{,}88$ con 173 grados de libertad. El programa muestra el valor p como **0** por el redondeo numérico; no significa que la probabilidad sea literalmente cero, sino que es extremadamente pequeña a la precisión mostrada. Según el modelo, se rechazaría $H_0: \sigma=8$ al 5 %.

**Implicación y limitación:** el contraste $\chi^2$ para una varianza requiere normalidad e independencia. La falta de normalidad que se evidencia en los gráficos y en Shapiro–Wilk debilita la validez del valor p. Por ello, el resultado sirve para practicar el contraste y reconocer sus supuestos, no como conclusión robusta sobre la dispersión poblacional.

In [ ]:
# Eliminamos los valores ausentes antes de comparar los cuantiles.
cuantiles, (pendiente, origen, _) = stats.probplot(datos['Bill'].dropna(), dist='norm')  # Calcula cuantiles normales teóricos y ajusta una recta a las facturas válidas.
teoricos, observados = cuantiles  # Separa las coordenadas teóricas y observadas del gráfico Q–Q.

# Mismo azul claro que el histograma anterior, con puntos más grandes y bordes negros.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico.
ax.scatter(teoricos, observados, s=65, color='#9fc3e4',  # Dibuja cada cuantil observado frente a su equivalente normal teórico.
           edgecolor='black', linewidth=0.8)  # Añade bordes para distinguir los puntos.
ax.plot(teoricos, pendiente * teoricos + origen, color='#800000', linewidth=1.8)  # Dibuja la recta de referencia ajustada.
ax.set_title('Gráfico de cuantiles de la factura', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Cuantiles teóricos de la distribución normal', fontsize=16)  # Etiqueta el eje horizontal.
ax.set_ylabel('Cuantiles observados de la factura', fontsize=16)  # Etiqueta el eje vertical.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el gráfico Q–Q.

### Interpretación del gráfico Q–Q

Si los importes siguieran una distribución normal, los puntos se situarían aproximadamente sobre la recta de referencia. En este gráfico se curvan y se apartan de ella, sobre todo en la cola superior: los importes más altos, hasta 300, quedan muy por encima de la recta.

**Implicación:** el patrón respalda visualmente la asimetría del histograma y la presencia de una cola derecha pesada. No es una desviación menor concentrada en un único punto; la forma general se aleja de la normalidad. Esto pone en duda el uso de intervalos y contrastes para la varianza basados en $\chi^2$.

### 2.2.3 Prueba de normalidad de Shapiro–Wilk

El gráfico de cuantiles permite valorar visualmente la normalidad. La **prueba de Shapiro–Wilk** la contrasta formalmente: compara la distribución de los datos con la que esperaríamos si procedieran de una población normal.

$$
\left\{
\begin{aligned}
H_0 &: \text{la distribución de las facturas es normal} \\
H_1 &: \text{la distribución de las facturas no es normal}
\end{aligned}
\right.
$$

El estadístico $W$ suele acercarse a 1 cuando los datos se ajustan a una distribución normal. Con un nivel de significación del 5 %, si el **valor p** es menor que $0{,}05$, rechazamos $H_0$; en caso contrario, no tenemos suficiente evidencia para rechazarla (**no** se demuestra que los datos sean normales). La prueba supone observaciones independientes y, como en el Q–Q plot, excluimos las facturas ausentes.

In [ ]:
# Contraste de normalidad de Shapiro–Wilk para las facturas sin valores ausentes.
w, p_valor = stats.shapiro(datos['Bill'].dropna())  # Calcula el estadístico W y el valor p de normalidad.
print(f'Shapiro–Wilk: W = {w:.4f}, valor p = {p_valor:.4g}')  # Muestra ambos resultados con redondeo legible.
if p_valor < 0.05:  # Comprueba si el valor p es menor que el nivel de significación del 5 %.
    print('Se rechaza la normalidad al nivel del 5 %.')  # Rechaza H₀ de normalidad cuando p < 0,05.
else:  # Este bloque se ejecuta cuando p es mayor o igual que 0,05.
    print('No se rechaza la normalidad al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar normalidad.

### Interpretación de Shapiro–Wilk

El estadístico es **$W=0{,}7336$** y el valor p es aproximadamente **$2{,}0\times10^{-16}$**, mucho menor que 0,05. Rechazamos la hipótesis de normalidad para estos datos. La prueba confirma lo que ya sugerían el histograma y las desviaciones marcadas del Q–Q plot.

**Qué significa y qué no:** hay evidencia de que la distribución de las facturas no es normal; la prueba no identifica la causa ni evalúa la representatividad de la muestra. Un valor p pequeño tampoco mide cuánto afecta la no normalidad a cada procedimiento. En este caso, la asimetría visible y los valores extremos hacen especialmente importante respetar la limitación de los métodos para varianza basados en normalidad.

### ¿Por qué usamos bootstrap?

Shapiro–Wilk y los gráficos indican que las facturas se alejan de la normalidad. Para estimar la incertidumbre de la desviación típica sin suponer una población normal, usamos **bootstrap**: extraemos muchas muestras del mismo tamaño a partir de los datos observados, con reemplazo, y calculamos la desviación típica en cada una. La distribución de esas estimaciones aproxima su variabilidad muestral; sus cuantiles forman el intervalo de confianza.

Bootstrap no crea información nueva ni corrige una muestra sesgada o dependiente. Su validez sigue dependiendo de que los datos observados representen razonablemente la población. En el siguiente bloque se generan las remuestras y se visualiza la distribución resultante.

<div style="break-after: page; page-break-after: always;"></div>

In [ ]:
rng = np.random.default_rng(12345)  # Fija la semilla para que las remuestras sean reproducibles.
bill_validos = datos['Bill'].dropna().to_numpy()  # Guarda las facturas no ausentes como un arreglo de NumPy.
m = np.array([np.std(rng.choice(bill_validos, size=len(bill_validos), replace=True), ddof=1)  # Remuestrea con reemplazo y calcula la desviación típica de cada remuestra.
              for _ in range(10_000)])  # Repite el remuestreo 10 000 veces y reúne las desviaciones en un arreglo.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y el eje para el histograma.
ax.hist(m, bins='auto', color='#9fc3e4', edgecolor='#666666', linewidth=1.2)  # Representa la distribución de las desviaciones bootstrap.
ax.set_title('Distribución bootstrap de la desviación típica', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Desviación típica bootstrap', fontsize=16)  # Etiqueta el eje horizontal.
ax.set_ylabel('Frecuencia', fontsize=16)  # Etiqueta el eje vertical con el número de remuestras.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el histograma bootstrap.

### Lectura del histograma bootstrap

Este gráfico representa la **variabilidad de la desviación típica** calculada en 10 000 remuestras del mismo tamaño, no la distribución de las facturas individuales. La mayor concentración se sitúa aproximadamente entre 35 y 45, aunque algunas remuestras producen estimaciones menores o mayores.

**Implicación:** los cuantiles de esta distribución aproximan la incertidumbre de la desviación típica poblacional bajo el remuestreo. Bootstrap no exige que la población sea normal, pero sí depende de que la muestra represente razonablemente a la población y de que las observaciones sean independientes. Repetir remuestras no corrige sesgo de selección, dependencia ni errores en los datos.

### Intervalo de confianza bootstrap para la desviación típica

El histograma anterior muestra **10 000 desviaciones típicas muestrales**: cada una se calcula a partir de una muestra del mismo tamaño que los datos válidos, elegida **con reemplazo**. Así aproximamos cómo varía la estimación al repetir el muestreo.

Para obtener el **IC bootstrap del 95 % por el método de los percentiles**, tomamos los percentiles **2,5 % y 97,5 %** de esas 10 000 desviaciones: los valores centrales quedan entre ambos límites. El intervalo estima la **desviación típica poblacional**, no el rango que contiene el 95 % de las facturas individuales. A diferencia del IC basado en χ² de más arriba, este procedimiento no necesita suponer normalidad; sí requiere que la muestra sea representativa y sus observaciones sean independientes.

In [ ]:
limite_inferior, limite_superior = np.quantile(m, [0.025, 0.975])  # Extrae los percentiles 2,5 y 97,5 como límites del IC bootstrap del 95 %.
print(f'Percentil 2,5 % (límite inferior): {limite_inferior:.2f}')  # Muestra el extremo inferior con dos decimales.
print(f'Percentil 97,5 % (límite superior): {limite_superior:.2f}')  # Muestra el extremo superior con dos decimales.
print(f'IC bootstrap del 95 % para la desviación típica poblacional: '  # Inicia la frase que identifica el intervalo y el parámetro estimado.
      f'[{limite_inferior:.2f}, {limite_superior:.2f}]')  # Completa e imprime los dos límites del intervalo.

### Interpretación del intervalo bootstrap

El IC percentil del 95 % para la desviación típica poblacional es aproximadamente **[30,26; 52,60]**. El intervalo es amplio, lo que refleja una incertidumbre apreciable en la estimación, coherente con la dispersión y la cola de importes altos.

**No confundir:** estos límites no contienen el 95 % de las facturas; estiman la desviación típica del conjunto poblacional. El método evita el supuesto de normalidad del intervalo basado en $\chi^2$, pero la calidad de la inferencia aún depende del diseño de muestreo y de la independencia. En la micropráctica, el IC bootstrap del 90 % debe ser más estrecho que el del 95 % porque sacrifica cobertura para ganar precisión.

#### Micropráctica: intervalo bootstrap del 90 %

Consulta los percentiles usados para el IC bootstrap del 95 % y adapta el par de cuantiles para obtener el 90 % central. Completa los dos huecos, ejecuta y compara la amplitud.

In [ ]:
try:
    # Construye el IC bootstrap del 90 % y compara su amplitud con el del 95 %.
    limites_90 = np.quantile(m, [..., ...])  # Selecciona los dos cuantiles que dejan el 90 % central.
    amplitud_90 = limites_90[1] - limites_90[0]  # Calcula la anchura del intervalo del 90 %.
    amplitud_95 = ...  # Calcula la anchura del IC del 95 % obtenido en la celda anterior.
    print(f'IC bootstrap del 90 %: [{limites_90[0]:.2f}, {limites_90[1]:.2f}]')  # Muestra sus límites.
    print(f'Amplitud 90 %: {amplitud_90:.2f}; amplitud 95 %: {amplitud_95:.2f}')  # Contrasta la precisión de ambos intervalos.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

## 2.3. Estimación de una proporción

Queremos estimar la proporción poblacional $\pi$ de estudiantes con tarjeta prepago. La proporción observada es $\hat{p}=e/n$, donde $e$ es el número de casos de prepago y $n$ el número de contratos informados. Los valores ausentes no forman parte del denominador. A continuación se presentan dos métodos de intervalo, ambos con confianza del 95 %; si `Plan` y `Prepaid` son las únicas categorías, la proporción de prepago es el complemento de la de plan fijo.

In [ ]:
e = (datos['PlanType'] == 'Prepaid').sum()  # Cuenta las filas con tarjeta prepago; las ausencias no cuentan como True.
n_plan = datos['PlanType'].notna().sum()  # Cuenta los estudiantes cuyo tipo de contrato está informado.
print(f'Prepago: {e} de {n_plan}; proporción observada = {e/n_plan:.4f}')  # Muestra el recuento y la proporción observada entre casos válidos.

### Interpretación del recuento

De los **174** estudiantes cuyo tipo de contrato está informado, **98** tienen `Prepaid`. La proporción observada es $98/174\approx0{,}563$, es decir, **56,3 %**.

**Por qué importa el denominador:** se excluyen los valores ausentes porque no sabemos a qué categoría pertenecen. Si se contaran como contratos distintos o como no-prepago, la proporción quedaría alterada. El 56,3 % es una estimación muestral; para inferir la proporción de toda la población necesitamos reflejar su incertidumbre con un intervalo.

### 2.3.1a Intervalo de confianza del 95 % para una proporción: método de Wilson

El intervalo de Wilson se obtiene invirtiendo el contraste score de la distribución binomial. Para confianza $1-\alpha$, se define $z=z_{1-\alpha/2}$, cuantil de la normal estándar, y $\hat{p}=e/n$:

$$
IC_{1-\alpha}^{\mathrm{Wilson}}(\pi)=\left[\frac{\hat{p}+\frac{z^2}{2n}}{1+\frac{z^2}{n}}-\frac{z}{1+\frac{z^2}{n}}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}+\frac{z^2}{4n^2}},\;\frac{\hat{p}+\frac{z^2}{2n}}{1+\frac{z^2}{n}}+\frac{z}{1+\frac{z^2}{n}}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}+\frac{z^2}{4n^2}}\right].
$$

El intervalo estima la proporción poblacional, no la proporción de una persona concreta. Wilson suele tener mejor comportamiento que el intervalo normal simple, especialmente cuando la proporción está cerca de 0 o 1. Para el 95 %, $\alpha=0{,}05$ y $z\approx1{,}96$.

In [ ]:
from statsmodels.stats.proportion import proportion_confint  # Importa funciones para intervalos de confianza de proporciones.

ic_wilson = proportion_confint(e, n_plan, alpha=0.05, method='wilson')  # Calcula el IC Wilson del 95 % con los casos y el total válidos.
print(f'IC Wilson del 95 % para la proporción de prepago: [{ic_wilson[0]:.4f}, {ic_wilson[1]:.4f}]')  # Muestra los límites del intervalo con cuatro decimales.

### Interpretación del IC de Wilson

El IC de Wilson del 95 % es aproximadamente **[0,4889; 0,6348]**, o entre **48,9 % y 63,5 %**. El intervalo incluye 0,50: una proporción poblacional de la mitad resulta compatible con estos datos al nivel de confianza del 95 %.

**Implicación:** aunque la estimación puntual sea 56,3 %, la incertidumbre no permite asegurar que la proporción poblacional supere el 50 %. El intervalo estima el parámetro poblacional, no predice la categoría de una persona concreta. La cobertura del 95 % se refiere al comportamiento del procedimiento en muestreos repetidos, no a una probabilidad posterior para este intervalo fijo.

#### Micropráctica: reconstruir un IC de Wilson

A partir del ejemplo de `Prepaid`, construye el intervalo para la misma categoría, pero calcula de nuevo los éxitos y el total a partir de `PlanType`. Adapta el nivel de confianza al 99 % y completa los argumentos de `proportion_confint`. Luego compara su amplitud con el IC del 95 %.

In [ ]:
try:
    # Reconstruye el IC Wilson del 99 % a partir de la columna PlanType.
    planes_validos = datos['PlanType'].dropna()  # Conserva solo contratos informados.
    exitos = (planes_validos == ...).sum()  # Cuenta los contratos de la categoría que se quiere estudiar.
    total = ...  # Calcula el número de estudiantes válidos.
    alpha = ...  # Convierte la confianza solicitada en el nivel alfa.
    ic_99 = proportion_confint(exitos, total, alpha=..., method=...)  # Completa los argumentos para usar el método Wilson.
    print(f'IC del 99 %: [{ic_99[0]:.4f}, {ic_99[1]:.4f}]')  # Muestra los límites.
    # Compáralo con el IC del 95 % y comenta su amplitud.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### 2.3.1b Intervalo binomial exacto de Clopper–Pearson

El intervalo exacto invierte las colas de la distribución binomial. Para $e$ éxitos entre $n$ observaciones y confianza $1-\alpha$, sus límites se expresan mediante cuantiles de la distribución beta:

$$
IC_{1-\alpha}^{\mathrm{CP}}(\pi)=\left[ B^{-1}\!\left(\frac{\alpha}{2};e,n-e+1\right),\;B^{-1}\!\left(1-\frac{\alpha}{2};e+1,n-e\right)\right],
$$

con límite inferior 0 si $e=0$ y límite superior 1 si $e=n$. Aquí $B^{-1}(q;a,b)$ es el cuantil $q$ de una distribución beta con parámetros $a,b$. El método no usa una aproximación normal y garantiza cobertura al menos nominal, por lo que suele ser conservador y algo más ancho que Wilson. Seguimos estimando la misma proporción, usando los mismos éxitos y el mismo total válido.

In [ ]:
ic_exacto = stats.binomtest(e, n_plan).proportion_ci(confidence_level=0.95, method='exact')  # Calcula el intervalo binomial exacto de Clopper–Pearson al 95 %.
print(f'IC binomial exacto del 95 % para la proporción de prepago: '  # Inicia el texto que identifica el intervalo.
      f'[{ic_exacto.low:.4f}, {ic_exacto.high:.4f}]')  # Añade e imprime los límites inferior y superior.

### Comparación con Clopper–Pearson

El intervalo exacto de Clopper–Pearson es **[0,4861; 0,6381]**, algo más ancho que el intervalo de Wilson **[0,4889; 0,6348]**. Los dos incluyen 0,50 y por tanto conducen a la misma lectura cualitativa: con estos datos no podemos descartar que la proporción poblacional sea la mitad.

**Implicación:** los métodos pueden dar límites ligeramente distintos porque construyen la cobertura de manera diferente. Clopper–Pearson suele ser conservador; en este ejemplo, la diferencia de anchura no cambia la conclusión. Ninguno elimina la necesidad de una muestra adecuada e independiente.

### 2.3.2 Contraste binomial: ¿es 0,5 la proporción de prepago?

Los intervalos **estiman** la proporción; este contraste bilateral pregunta si los datos son compatibles con que **la mitad** de los estudiantes con contrato conocido tenga tarjeta prepago:

$$
\begin{aligned}
H_0 &: \pi = 0{,}5 \\
H_1 &: \pi \ne 0{,}5
\end{aligned}
$$

El estadístico es el número de éxitos observados, $X=e$. Bajo la hipótesis nula, $X\sim\operatorname{Binomial}(n,\pi_0)$; aquí se observan $e=98$ éxitos de $n=174$ y $\hat{p}=e/n\approx0{,}5632$. La prueba exacta obtiene el valor p a partir de esa distribución discreta, sin aproximar $X$ por una normal. Para el contraste bilateral de SciPy, se suman las probabilidades de los resultados cuya probabilidad bajo $H_0$ es igual o menor que la del resultado observado.

Aquí $\pi$ es la proporción **poblacional** de prepago. Si el valor p es menor que $0{,}05$, rechazamos $H_0$; de lo contrario, **no la rechazamos** (no significa que $\pi$ sea exactamente 0,5). Se supone que las observaciones son independientes y que cada estudiante tiene la misma probabilidad de ser prepago.

In [ ]:
resultado_binomial = stats.binomtest(e, n_plan, p=0.5, alternative='two-sided')  # Contrasta exactamente si la proporción poblacional de prepago es 0,5, en ambas direcciones.
print(f'Valor p del contraste binomial bilateral (H₀: π = 0,5): {resultado_binomial.pvalue:.4g}')  # Muestra el valor p que se compara con alfa.
if resultado_binomial.pvalue < 0.05:  # Comprueba si el valor p es menor que el umbral del 5 %.
    print('Se rechaza H₀ al nivel del 5 %.')  # Rechaza la hipótesis de proporción 0,5 si el resultado es significativo.
else:  # Se ejecuta si el valor p es al menos 0,05.
    print('No se rechaza H₀ al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar la hipótesis nula.

### Interpretación del contraste binomial

El valor p es **0,111**, mayor que $\alpha=0{,}05$; por tanto, **no rechazamos $H_0: \pi=0{,}5$**. No hay evidencia estadística suficiente para afirmar que la proporción poblacional de prepago difiera de la mitad, aunque la proporción observada sea 56,3 %.

La conclusión concuerda con los dos intervalos del 95 %, que incluyen 0,50. **No rechazar no demuestra** que la proporción sea exactamente 0,50: significa que, con esta muestra y este nivel, no se detecta una diferencia suficiente. La conclusión presupone observaciones independientes y un muestreo apropiado.

#### Micropráctica: contrastar otra proporción de referencia

Revisa la llamada de `stats.binomtest` de la celda anterior. Completa `p_ref` con 0,60, ejecuta el contraste bilateral y compara su valor p con el obtenido para 0,50. Decide al nivel del 5 % y expresa la conclusión con cautela.

In [ ]:
try:
    # Repite el contraste binomial cambiando la hipótesis nula a π = 0,60.
    p_ref = ...  # Escribe el valor de referencia para la proporción poblacional.
    resultado = stats.binomtest(..., ..., p=..., alternative=...)  # Completa casos, tamaño, referencia y dirección del contraste.
    p_valor_nuevo = ...  # Extrae el valor p del resultado.
    print(f'p-valor para H₀: π = {p_ref:.2f}: {p_valor_nuevo:.4g}')  # Muestra el resultado.
    if p_valor_nuevo < ...:  # Aplica el nivel de significación del 5 %.
        print('Se rechaza H₀ al nivel del 5 %.')
    else:
        print('No se rechaza H₀ al nivel del 5 %.')
    # Compara la decisión con el contraste anterior para π = 0,50.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

## Ejercicio 2.1 — Inferencia con datos de vuelos

Analiza los retrasos de llegada del conjunto `flights.txt`. Completa los cuatro apartados en orden: primero carga y valida los datos; después estima la media, estudia la distribución y estima la desviación típica mediante bootstrap; por último, calcula la proporción de vuelos que llegan tarde. Usa en cada apartado el nivel de confianza indicado y redacta las conclusiones en contexto.

### Antes de empezar

Resuelve cada apartado de forma independiente y en orden. Usa los resultados de las celdas anteriores como modelo, pero adapta el método a la variable y al nivel de confianza que se piden. Después de ejecutar cada bloque, redacta una conclusión en el contexto de los vuelos; no basta con copiar los números.

#### 2.1.1 — Cargar y revisar los vuelos

Lee `flights.txt` desde `DATA_PATH`, separado por tabulaciones. Trata `@`, `NA` y las celdas vacías como valores ausentes. Guarda la tabla en `vuelos_df`; comprueba las primeras filas, los nombres de columnas y el número de ausentes antes de continuar.

In [ ]:
# Pista: pd.read_csv(DATA_PATH / 'flights.txt', sep='\t', na_values=['@', 'NA', '']).
# Guarda la tabla en vuelos_df y revisa la carga con display(vuelos_df.head()).
# Después consulta vuelos_df.info() o vuelos_df.isna().sum() para ver tipos y ausentes.
# Sustituye pass por tu solución.
pass

#### 2.1.2 — Intervalo del 95 % para la media del retraso

Trabaja con `vuelos_df['arr_delay']` y elimina los valores ausentes. Aplica la fórmula t de Student para la media desarrollada en la sección 1.1, o reutiliza `intervalo_media(..., confianza=0.95)`, e indica las unidades. Comprueba si el intervalo incluye 0; un retraso medio negativo corresponde a llegadas anticipadas.

In [ ]:
# Pista: selecciona arr_delay y elimina ausentes con dropna().
# Usa intervalo_media(serie, confianza=0.95) y muestra el resultado.
# En la conclusión, interpreta el intervalo para la media poblacional, no para un vuelo individual.
# Sustituye pass por tu solución.
pass

#### 2.1.3 — Revisar la normalidad y estimar la desviación típica

Grafica el histograma y el Q–Q plot de `arr_delay` sin ausentes. Describe las desviaciones respecto de la normalidad. Después, realiza remuestreo bootstrap con reemplazo y calcula una desviación típica en cada remuestra usando `ddof=1`. Aplica el intervalo percentil explicado arriba: para el 90 %, toma los cuantiles $Q_{0{,}05}$ y $Q_{0{,}95}$. Usa una semilla para que el resultado sea reproducible.

In [ ]:
# Pistas:
# - Usa serie = vuelos_df['arr_delay'].dropna().
# - Para diagnosticar forma, combina ax.hist(...) y stats.probplot(serie, dist='norm').
# - Para bootstrap, repite rng.choice(serie, size=len(serie), replace=True) y
#   calcula np.std(remuestra, ddof=1); guarda las estimaciones en un arreglo.
# - IC central del 90 %: cuantiles 0.05 y 0.95 mediante np.quantile(...).
# Escribe aquí el análisis y comenta supuestos y limitaciones.

#### 2.1.4 — Proporción de vuelos con retraso e IC del 99 %

Define como retrasados los vuelos con `arr_delay > 0`. Excluye primero los retrasos ausentes; luego cuenta éxitos $e$ y total válido $n$. Aplica Wilson —fórmula desarrollada arriba— con $\alpha=0{,}01$ y $z=z_{0{,}995}$, o usa Clopper–Pearson con confianza 0,99. Explica qué proporción poblacional estima el intervalo.

In [ ]:
# Pistas:
# - Elimina ausentes de arr_delay antes de calcular éxitos y total.
# - Cuenta retrasos con la condición arr_delay > 0.
# - Calcula proportion_confint(exitos, total, alpha=0.01, method='wilson')
#   o usa stats.binomtest(...).proportion_ci(confidence_level=0.99, method='exact').
# Presenta límites y expresa qué proporción poblacional estiman.

<div style="break-after: page; page-break-after: always;"></div>

## Reto final integrador (opcional)

Analiza la factura de los estudiantes del grupo `Plan`. Resuelve los apartados en orden, reutilizando las funciones anteriores y redactando una conclusión propia. No copies celdas de resultados.

### 1. Seleccionar el subgrupo

Filtra las filas cuyo grupo sea `Plan` y conserva las facturas no ausentes de `Bill`. Comprueba el tamaño del subgrupo y utiliza una serie limpia para los análisis siguientes.

In [ ]:
# Pista: selecciona datos.loc[datos['PlanType'] == 'Plan', 'Bill'] y aplica dropna().
# Guarda la serie limpia como serie_plan y comprueba cuántos valores válidos contiene.
# Sustituye pass por tu código.
pass

### 2. Estimar y contrastar la media

Calcula el IC bilateral del 99 % para la media del subgrupo y contrasta $H_0: \mu=50$ frente a una alternativa bilateral. Compara si el intervalo contiene 50 con la decisión del contraste usando $\alpha=0{,}01$. Explica si las conclusiones coinciden.

In [ ]:
# Pistas: reutiliza intervalo_media(serie_plan, confianza=0.99).
# Después usa contraste_media(serie_plan, mu=50) y compara con alpha = 0.01.
# Muestra ambos resultados y escribe una conclusión en contexto.

### 3. Revisar normalidad y concluir

Comprueba la forma de la distribución con un Q–Q plot frente a la normal. Indica si la normalidad parece razonable y cómo afecta a la confianza en los métodos usados. Cierra con una conclusión que incluya tamaño muestral, estimación, IC, contraste, nivel de significación y limitaciones de representatividad.

In [ ]:
# Pistas: usa stats.probplot(serie_plan, dist='norm') y representa los puntos junto con la recta.
# Describe las desviaciones visibles y relaciona el diagnóstico con los supuestos del contraste.
# Añade aquí tu conclusión final en contexto.